In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common import *
import yaml
import matplotlib.colors as mcolors

import library

In [ ]:
device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette()
colors_set = sns.color_palette("Paired", n_colors=20)
colors

default plot setting

In [ ]:
colors_set

In [ ]:
color_continuas = sns.color_palette("mako", n_colors=8)
color_continuas

In [ ]:
with open("../linear/config.yaml", 'r', encoding='utf-8') as f:
    configs_linear = yaml.safe_load(f)
with open("../deep/config.yaml", 'r', encoding='utf-8') as f:
    configs_deep = yaml.safe_load(f)
with open("../mlp/config.yaml", 'r', encoding='utf-8') as f:
    configs_mlp = yaml.safe_load(f)

config = configs_mlp

In [ ]:
def sample(batch, sizes, device, mul=0.01):
    x_out = torch.zeros((batch, 0), device=device)
    for size in sizes:
        rand = torch.rand((batch, 1), device=device).round() * 2 - 1
        x_out = torch.cat([x_out, rand.repeat(1, size)], dim=-1)
    epsilon1 = mul * torch.randn_like(x_out)
    epsilon2 = mul * torch.randn_like(x_out)
    return x_out + epsilon1, x_out + epsilon2

sample makes [+1, -1] datas

In [ ]:
def create_segment_vectors(sizes):
    total_length = sum(sizes)
    cumsum = torch.cumsum(torch.tensor(sizes), dim=-1)

    result = torch.zeros((len(sizes), total_length))

    for i in range(len(sizes)):
        start_idx = cumsum[i - 1] if i > 0 else 0
        end_idx = cumsum[i]

        result[i, start_idx:end_idx] = 1

    return result

In [ ]:
batch = config["batch"]
sizes = config["sizes"]
s_in = sum(sizes)
s_out = 2

In [ ]:
sample_data = sample(batch, sizes, device)
gamma = torch.bmm(sample_data[0].unsqueeze(dim=2), sample_data[1].unsqueeze(dim=1))  # b x n x n
gamma = gamma + gamma.permute(0, 2, 1)
gamma = torch.mean(gamma, dim=0).squeeze() / 2  # n x n
sns.heatmap(gamma.cpu())
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)

In [ ]:
loss_types = "BarlowTwins_test_relu"

e = create_segment_vectors(sizes).to(device)

lambdas_render_barlow = ["0", "0.1", "0.5", "1"]
# lambdas_render_barlow = ["1"]
output_dict = np.load("data.npy", allow_pickle=True).item()

In [ ]:
t0 = 13000
epochs = config[loss_types]["epochs"]

if "BarlowTwins" in loss_types:
    lambdas_render = ["1"]
else:
    lambdas_render = [1]
for lambd in lambdas_render:
    if "BarlowTwins" in loss_types:
        output_target = output_dict[loss_types][lambd]
    else:
        output_target = output_dict[loss_types]
    losses = output_target["loss"]
    eiglist = output_target["eiglist"]
    feature_alignment = output_target["feature_alignment"]
    s0 = output_target["s0"]

    dldt = smooth_data(dt(losses), 15)
    # plt.plot(dldt)

    fig, axs = plt.subplots(1, 1, figsize=(8, 4))

    for i in range(len(eiglist)):
        axs.plot(eiglist[i], linewidth=6, color=colors[i], alpha=0.5,
                    label=(r"$\lambda_" + ("l$" if i == 0 else "s$")))
    # for pos in critical_pos:
    #     axs.axvline(pos, color='black', linestyle='--')
    # axs[1].set_ylim(-0.05, 1.10)
    axs.set_xlabel("Step")
    axs.set_ylabel("Eigenvalue")
    axs.legend()

    plt.tight_layout()
    plt.savefig("figure/eig_relu.pdf", format="pdf", bbox_inches='tight')
    plt.show()

In [ ]:
t0 = 13000
epochs = config[loss_types]["epochs"]

if "BarlowTwins" in loss_types:
    lambdas_render = ["1"]
else:
    lambdas_render = [1]
for lambd in lambdas_render:
    if "BarlowTwins" in loss_types:
        output_target = output_dict[loss_types][lambd]
    else:
        output_target = output_dict[loss_types]
    losses = output_target["loss"]

    fig, axs = plt.subplots(1, 1, figsize=(10, 5))
    axs.plot(losses, color=colors[1], linewidth=5, alpha=0.9, label=(r"$\lambda = " + f"{lambd}" + "$"))

    axs.set_ylim(-0.05, s_out * 1.15)
    axs.set_xlabel("Step")
    axs.set_ylabel("Loss")

    plt.tight_layout()
    plt.savefig("figure/loss_only.pdf", format="pdf", bbox_inches='tight')
    plt.show()

In [ ]:
v1_dict = np.load("../deep/data.npy", allow_pickle=True).item()
v2_dict = np.load("../mlp/data.npy", allow_pickle=True).item()
v1_target = v1_dict[loss_types]["0"]
slope_list = configs_mlp["BarlowTwins_test_leakyrelu"]["slope"]
list.reverse(slope_list)
v3_target = v2_dict["BarlowTwins_test_relu"]["0"]

target_list = [v1_target, *[v2_dict["BarlowTwins_test_leakyrelu"]["0"][key] for key in slope_list], v3_target]
name_list = ["Deep", *[f"LeakyRelu(slope={key})" for key in slope_list], "Relu"]

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(20, 12))
for i in range(len(name_list)):
    # for j in range(len(target_list[i]['eiglist'])):
    if "Deep" in name_list[i]:
        axs[0].plot(target_list[i]['loss'], color=color_continuas[0], alpha=1.0, linewidth=2.5, label=name_list[i], linestyle='--')
    elif "Relu" == name_list[i]:
        axs[0].plot(target_list[i]['loss'], color=color_continuas[-1], alpha=1.0, linewidth=2.5, label=name_list[i], linestyle='--')
    else:
        axs[0].plot(target_list[i]['loss'], color=color_continuas[i], alpha=0.6, linewidth=7.0, label=name_list[i])
axs[0].legend(loc='upper right')
axs[0].set_xlabel('Step')
axs[0].set_title('Loss')
for i in range(len(name_list)):
    # for j in range(len(target_list[i]['eiglist'])):
    print(f"{target_list[i]['eiglist'][0][0]:.7f}")
    if "Deep" in name_list[i]:
        axs[1].plot(target_list[i]['eiglist'][0], color=color_continuas[0], alpha=1.0, linewidth=2.5, label=name_list[i], linestyle='--')
    elif "Relu" == name_list[i]:
        axs[1].plot(target_list[i]['eiglist'][0], color=color_continuas[-1], alpha=1.0, linewidth=2.5, label=name_list[i], linestyle='--')
    else:
        axs[1].plot(target_list[i]['eiglist'][0], color=color_continuas[i], alpha=0.6, linewidth=7.0, label=name_list[i])
axs[1].legend(loc='lower right')
axs[1].set_xlabel('Step')
axs[1].set_title(r'Eigenvalue $\lambda_l$')
# plt.yscale('log')
plt.tight_layout()
plt.savefig('figure/barlowtwins/non_linearity.pdf', format="pdf")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(8, 4))

for i in range(len(name_list)):
    # for j in range(len(target_list[i]['eiglist'])):
    print(f"{target_list[i]['eiglist'][0][0]:.7f}")
    if "Deep" in name_list[i]:
        axs[1].plot(target_list[i]['eiglist'][0], color=color_continuas[0], alpha=1.0, linewidth=2.5, label=name_list[i], linestyle='--')
    elif "Relu" == name_list[i]:
        axs[1].plot(target_list[i]['eiglist'][0], color=color_continuas[-1], alpha=1.0, linewidth=2.5, label=name_list[i], linestyle='--')
    else:
        axs[1].plot(target_list[i]['eiglist'][0], color=color_continuas[i], alpha=0.6, linewidth=7.0, label=name_list[i])
axs[1].legend(loc='lower right')
axs[1].set_xlabel('Step')
axs[1].set_title(r'Eigenvalue $\lambda_l$')
# plt.yscale('log')
plt.tight_layout()
plt.savefig('figure/barlowtwins/non_linearity.pdf', format="pdf")
plt.show()